In [15]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import torch

In [16]:
messages = [
        {
            "role": "system",
            "content": "You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe.  Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.\n\nIf a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information."
        },
]

In [17]:
data_path = "/data/shenggui/projects/spec-decoding/EAGLE/eagle/traineagle3/sharegpt_expanded.jsonl"
with open(data_path, "r") as f:
    for line in f:
        data = json.loads(line)
        print(data)
        break


{'id': 'QWJhYvA_0', 'conversations': [{'role': 'user', 'content': "Summarize the main ideas of Jeff Walker's Product Launch Formula into bullet points as it pertains to a growth marketing agency implementing these strategies and tactics for their clients..."}, {'role': 'assistant', 'content': "Here are the main ideas of Jeff Walker's Product Launch Formula that can be applied by a growth marketing agency for their clients:\n\n1. Identify the target audience and their needs: Understand the ideal customer for the product or service, and create a messaging that resonates with them.\n2. Pre-launch: Build anticipation and excitement for the launch by creating buzz, gathering testimonials and case studies, and using social media to create awareness.\n3. Launch: Use a well-crafted launch sequence to maximize sales and conversions. This can include offering bonuses, creating scarcity, and using a deadline to create urgency.\n4. Post-launch: Follow up with customers, gather feedback, and contin

In [18]:
source = data['conversations']

if source[0]["role"] != "user":
    # Skip the first one if it is not from human
    source = source[1:]


for j, sentence in enumerate(source):
    role = sentence["role"]
    # if sentence["from"]=="gpt":
    #     sentence["value"]=" "+sentence["value"]
    messages.append(
        {"role": role, "content": sentence["content"]}
    )

print(messages)


[{'role': 'system', 'content': "You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe.  Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.\n\nIf a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information."}, {'role': 'user', 'content': "Summarize the main ideas of Jeff Walker's Product Launch Formula into bullet points as it pertains to a growth marketing agency implementing these strategies and tactics for their clients..."}, {'role': 'assistant', 'content': "Here are the main ideas of Jeff Walker's Product Launch Formula that can be applied by a growth marketing agency for their clients:\n\n1. Identify the target audience and their needs: Understand th

In [21]:
tokenizer = AutoTokenizer.from_pretrained("/root/.cache/huggingface/hub/models--Qwen--Qwen3-30B-A3B-Instruct-2507/snapshots/0d7cf23991f47feeb3a57ecb4c9cee8ea4a17bfe/")
template = "qwen"

conversation = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

if not tokenizer.pad_token_id:
    tokenizer.pad_token_id = tokenizer.unk_token_id

print(conversation)

input_ids = tokenizer(
    conversation,
    return_tensors="pt",
    max_length=4096,
    add_special_tokens=False,
).input_ids[0]

print(input_ids)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


<|im_start|>system
You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe.  Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, please don't share false information.<|im_end|>
<|im_start|>user
Summarize the main ideas of Jeff Walker's Product Launch Formula into bullet points as it pertains to a growth marketing agency implementing these strategies and tactics for their clients...<|im_end|>
<|im_start|>assistant
Here are the main ideas of Jeff Walker's Product Launch Formula that can be applied by a growth marketing agency for their clients:

1. Identify the target audience and their needs: Understand the ideal customer for the pr

In [22]:
loss_mask = torch.ones_like(input_ids)
# print(i)
import re

if template == "llama":
    assistant_message_separator = "<|start_header_id|>assistant<|end_header_id|>\n\n"
    end_of_turn_token = "<|eot_id|>"
else:
    assistant_message_separator = "<|im_start|>assistant\n"
    sep2 = "<|im_end|>\n"


assistant_pattern = (
            re.escape(assistant_message_separator)
            + r"([\s\S]*?(?:"
            + re.escape(end_of_turn_token)
            + "|$))"
        )

In [ ]:
# get input_ids
loss_mask = torch.zeros(len(input_ids), dtype=torch.long)
for match in re.finditer(assistant_pattern, conversation, re.DOTALL):
    content_start_char = match.start(1)
    content_end_char = match.end(1)

    # --- Core Alternative Operation: Calculate Token Index Based on Prefix String Length ---
    # Encode the text "assistant start", the length of which is the position of the starting token.
    prefix_ids = tokenizer.encode(
        conversation[:content_start_char], add_special_tokens=False
    )
    # Encodes the text "assistant end", the length of which is the position of the end token.
    full_ids = tokenizer.encode(
        conversation[:content_end_char], add_special_tokens=False
    )

    start_token_idx = len(prefix_ids)
    end_token_idx = len(full_ids)

    # Handling out-of-bounds errors caused by truncation
    actual_start = min(start_token_idx, len(input_ids))
    actual_end = min(end_token_idx, len(input_ids))

    if actual_start < actual_end:
        loss_mask[actual_start:actual_end] = 1


: 